# Mueller Matrix Inference: Prediction vs Ground Truth Comparison

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torchvision.models as models
from pathlib import Path
#    'Day6_mm_results_Day6E_4B_S2',
#    'Day6_mm_results_day6_3',
#    'Day0_mm_results_Day0G_7B_S3',
#    'Day0_mm_results_Day0H_3B_S6'
# ============================================================================
# 1. Configuration
# ============================================================================
class InferenceConfig:
    # Path to a sample that HAS masks (e.g., from your isolated list or test set)
    NPZ_PATH = Path(r"F:\MPL_Data\mmNoTissueFilter\TRIMMM\Day0\mm_results_Day0H_3B_S6\TRIMMM.npz")
    MODEL_PATH = Path("../../models/best_model.pth")
    OUTPUT_DIR = Path("inference_output")

    ENCODER_NAME = 'resnet34'
    INPUT_SIZE = (512, 512)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

InferenceConfig.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 2. Model Definition

In [ ]:
class UNetWithPretrainedEncoder(nn.Module):
    def __init__(self, encoder_name='resnet34', num_classes=2):
        super().__init__()
        encoder = models.resnet34(weights=None)
        self.encoder0 = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.encoder1 = nn.Sequential(encoder.maxpool, encoder.layer1)
        self.encoder2 = encoder.layer2
        self.encoder3 = encoder.layer3
        self.encoder4 = encoder.layer4

        enc_ch = [64, 64, 128, 256, 512]
        self.decoder4 = self._block(enc_ch[4]+enc_ch[3], enc_ch[3])
        self.decoder3 = self._block(enc_ch[3]+enc_ch[2], enc_ch[2])
        self.decoder2 = self._block(enc_ch[2]+enc_ch[1], enc_ch[1])
        self.decoder1 = self._block(enc_ch[1]+enc_ch[0], enc_ch[0])
        self.decoder0 = self._block(enc_ch[0], 64)
        self.final_conv = nn.Conv2d(64, num_classes, 1)

    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e0 = self.encoder0(x)
        e1 = self.encoder1(e0)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        d4 = self.decoder4(torch.cat([F.interpolate(e4, size=e3.shape[2:], mode='bilinear'), e3], 1))
        d3 = self.decoder3(torch.cat([F.interpolate(d4, size=e2.shape[2:], mode='bilinear'), e2], 1))
        d2 = self.decoder2(torch.cat([F.interpolate(d3, size=e1.shape[2:], mode='bilinear'), e1], 1))
        d1 = self.decoder1(torch.cat([F.interpolate(d2, size=e0.shape[2:], mode='bilinear'), e0], 1))
        return self.final_conv(self.decoder0(F.interpolate(d1, size=x.shape[2:], mode='bilinear')))


# 3. Data Loading (Extracts GT Masks)

In [ ]:
def load_data_with_gt(npz_path):
    """Loads M11 and constructs the Ground Truth mask (if available)."""
    try:
        with np.load(npz_path, allow_pickle=True) as data:
            # --- Load M11 ---
            if 'nM11s' in data: m11 = np.array(data['nM11s'])
            elif 'M11s' in data:
                raw = np.array(data['M11s'])
                m11 = (raw - raw.min()) / (raw.max() - raw.min()) if raw.max() > raw.min() else raw
            else: return None, None, "No M11 found"

            # --- Load Masks ---
            # 1. Tissue Mask
            tissue_mask = None
            for key in ['tissue_mask', 'annotation_mask']:
                if key in data: tissue_mask = np.array(data[key]) > 0; break

            if tissue_mask is None: return m11, None, "No tissue mask found"

            # 2. OS Mask
            os_mask = np.array(data['os_mask']) > 0 if 'os_mask' in data else None
            
            # 3. Vaginal Mask - check multiple possible keys
            vag_mask = None
            for key in ['vaginal_mask', 'vaginal_wall', 'vaginal_wall_mask']:
                if key in data:
                    vag_mask = np.array(data[key]) > 0
                    break

            # --- Combine into GT (Priority: Vaginal > OS > Tissue) ---
            gt_mask = np.zeros_like(tissue_mask, dtype=np.int64)
            gt_mask[tissue_mask] = 1       # Class 1: Tissue
            if os_mask is not None:
                gt_mask[os_mask] = 2       # Class 2: OS
            if vag_mask is not None:
                gt_mask[vag_mask] = 3      # Class 3: Vaginal

            return m11, gt_mask, "Success"

    except Exception as e:
        return None, None, str(e)

# 4. Run Comparison

In [ ]:
# 1. Load Data
m11, gt_mask, status = load_data_with_gt(InferenceConfig.NPZ_PATH)
if m11 is None: raise ValueError(f"Data load failed: {status}")

# 2. Load Model & Predict
checkpoint = torch.load(InferenceConfig.MODEL_PATH, map_location=InferenceConfig.DEVICE, weights_only=False)
model = UNetWithPretrainedEncoder(num_classes=4).to(InferenceConfig.DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded successfully")
print(f"  Training Val Loss: {checkpoint.get('val_loss', 'N/A'):.4f}" if 'val_loss' in checkpoint else "  Training Val Loss: N/A")
print(f"  Training Val Acc: {checkpoint.get('val_acc', 'N/A'):.4f}" if 'val_acc' in checkpoint else "  Training Val Acc: N/A")

# 3. Preprocess using ImageNet normalization (matching training)
H0, W0 = m11.shape

# Step 1: Min-max normalization (already done in load_data_with_gt)
# Step 2: Resize to 512x512
img_tensor = torch.from_numpy(m11).float().unsqueeze(0).unsqueeze(0)
img_tensor = F.interpolate(img_tensor, size=InferenceConfig.INPUT_SIZE, mode='bilinear', align_corners=True)

# Step 3: Replicate to 3 channels
img_tensor = img_tensor.repeat(1, 3, 1, 1)  # [1, 3, 512, 512]

# Step 4: ImageNet mean/std normalization
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
img_tensor = (img_tensor - imagenet_mean) / imagenet_std

# 4. Predict
with torch.no_grad():
    logits = model(img_tensor.to(InferenceConfig.DEVICE))
    pred_mask = torch.argmax(logits, dim=1)
    # Step 6: Resize back to original size
    pred_mask = F.interpolate(pred_mask.unsqueeze(1).float(), size=(H0, W0), mode='nearest')
    pred_mask = pred_mask.squeeze().cpu().numpy().astype(int)

print(f"\nInference complete!")
print(f"  Input shape: {m11.shape}")
print(f"  Output shape: {pred_mask.shape}")

# 5. Comprehensive Metrics & Visualization

**Metrics Calculated:**
- Overall Pixel Accuracy
- Per-Class IoU (Intersection over Union)
- Per-Class Dice Coefficient
- Per-Class Precision and Recall

In [ ]:
# Define Colors: 0=Black, 1=Blue, 2=Green, 3=Red
cmap_colors = ['black', 'blue', 'lime', 'red']
cmap = mcolors.ListedColormap(cmap_colors)
norm = mcolors.BoundaryNorm([0, 1, 2, 3, 4], cmap.N)

# ==================== COMPREHENSIVE METRICS ====================
classes = {0: "Background", 1: "Tissue", 2: "OS", 3: "Vaginal"}

# Overall Accuracy
overall_accuracy = (pred_mask == gt_mask).mean()

print(f"\n{'='*70}")
print(f"COMPREHENSIVE METRICS: {InferenceConfig.NPZ_PATH.parent.name}")
print(f"{'='*70}")
print(f"\n{'Overall Pixel Accuracy:':<30} {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")
print(f"\n{'Class':<15} {'GT Pixels':<12} {'Pred Pixels':<12} {'IoU':<10} {'Dice':<10} {'Precision':<10} {'Recall':<10}")
print("-" * 85)

# Per-class metrics
for c_id, c_name in classes.items():
    gt_c = (gt_mask == c_id)
    pred_c = (pred_mask == c_id)

    # IoU
    intersection = (gt_c & pred_c).sum()
    union = (gt_c | pred_c).sum()
    iou = intersection / union if union > 0 else 0.0
    
    # Dice
    dice = (2.0 * intersection) / (gt_c.sum() + pred_c.sum()) if (gt_c.sum() + pred_c.sum()) > 0 else 0.0
    
    # Precision and Recall
    tp = (gt_c & pred_c).sum()
    fp = (~gt_c & pred_c).sum()
    fn = (gt_c & ~pred_c).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    print(f"{c_name:<15} {gt_c.sum():<12} {pred_c.sum():<12} {iou:<10.4f} {dice:<10.4f} {precision:<10.4f} {recall:<10.4f}")

print(f"{'='*70}\n")

# --- PLOTTING ---
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
plt.suptitle(f"Sample: {InferenceConfig.NPZ_PATH.parent.name}\nOverall Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)", 
             fontsize=16, fontweight='bold')

# Row 1: General Overview
# 1. Original M11
axes[0,0].imshow(m11, cmap='gray')
axes[0,0].set_title("Original M11")
axes[0,0].axis('off')

# 2. Ground Truth
im_gt = axes[0,1].imshow(gt_mask, cmap=cmap, norm=norm, interpolation='nearest')
axes[0,1].set_title("Ground Truth Mask\n(Blue=Tissue, Green=OS, Red=Vaginal)")
axes[0,1].axis('off')

# 3. Prediction
im_pred = axes[0,2].imshow(pred_mask, cmap=cmap, norm=norm, interpolation='nearest')
axes[0,2].set_title("Model Prediction")
axes[0,2].axis('off')

# Row 2: Specific Error Analysis
# 4. Error Map
error_map = (gt_mask != pred_mask)
axes[1,0].imshow(error_map, cmap='Reds', vmin=0, vmax=1)
axes[1,0].set_title(f"Difference Map (Red = Error)\nAccuracy: {overall_accuracy:.2%}")
axes[1,0].axis('off')

# 5. OS Comparison (Binary)
# Overlay: Green=GT, Red=Pred, Yellow=Overlap
os_viz = np.zeros((*m11.shape, 3))
os_viz[gt_mask == 2, 1] = 1.0  # Green channel for GT
os_viz[pred_mask == 2, 0] = 1.0 # Red channel for Pred
# (Result: Green=Missed, Red=False Alarm, Yellow=Correct Match)
axes[1,1].imshow(os_viz)
axes[1,1].set_title("OS Comparison\n(Green=GT only, Red=Pred only, Yellow=Match)")
axes[1,1].axis('off')

# 6. Vaginal Comparison (Binary)
vag_viz = np.zeros((*m11.shape, 3))
vag_viz[gt_mask == 3, 1] = 1.0
vag_viz[pred_mask == 3, 0] = 1.0
axes[1,2].imshow(vag_viz)
axes[1,2].set_title("Vaginal Comparison\n(Green=GT only, Red=Pred only, Yellow=Match)")
axes[1,2].axis('off')

plt.tight_layout()
plt.savefig(InferenceConfig.OUTPUT_DIR / "comparison_vs_gt.png", dpi=150)
plt.show()

print(f"Saved comparison plot to: {InferenceConfig.OUTPUT_DIR / 'comparison_vs_gt.png'}")

In [ ]:
print("done")